# Lab 1 · Agent สี่แบบใน environment เดียวกัน

ใน lecture เราเรียนว่า agent คือสิ่งที่รับ percept จาก environment แล้วเลือก action และ agent ที่ rational คือตัวที่ทำคะแนนตาม performance measure ได้ดีที่สุด Lab นี้ทำให้แนวคิดนั้นเป็น code: คุณจะสร้าง agent 4 แบบ ปล่อยลงใน environment เดียวกัน แล้วเทียบคะแนน

**Environment: กล่องเอกสารขาเข้า** เอกสาร 24 ฉบับเข้ามาทีละฉบับ agent ต้องเลือกว่าเป็นเอกสารประเภทใด หรือส่งให้คนตรวจ

| PEAS | ในงานนี้ |
|---|---|
| **P**erformance | ถูก +1, ส่งให้คน 0, ผิด −1 |
| **E**nvironment | เอกสารธุรกิจภาษาไทยและอังกฤษที่เข้ามาตามลำดับ |
| **A**ctuators | เลือก 1 ใน 7 action: 6 ประเภทเอกสาร หรือ `needs_human` |
| **S**ensors | ข้อความของเอกสารฉบับปัจจุบัน |

**เวลา 70 นาที** · ส่วนที่ 1–6 เป็น Core ทุกคนต้องทำให้จบ · ส่วนที่ 7 เป็น Stretch

มองหา `# TODO` ใน code นั่นคือส่วนที่คุณต้องเขียน ทุก cell รันได้แม้ยังไม่แก้ แต่คะแนนจะต่ำ ก่อนเริ่มให้บันทึกไฟล์นี้เป็น `lab1_inbox_agents.mine.ipynb`

## 1. รู้จัก environment (5 นาที)

`InboxEnvironment` ส่งเอกสารให้ agent ทีละฉบับตามลำดับ แล้วให้คะแนนทุก action สิ่งที่ agent เห็นคือ `Percept` ซึ่งมีแค่ `doc_id` กับ `text` agent ไม่เห็นเฉลย

In [2]:
import pandas as pd

from agentlab import get_llm, release_model
from agentlab.inbox import ACTIONS, InboxEnvironment, summarize

env = InboxEnvironment()
print("documents:", len(env.percepts))
print("actions:  ", ACTIONS)
print("rewards:  ", env.rewards)

first = env.percepts[0]
print(f"\n--- {first.doc_id} ---\n{first.text}")

documents: 24
actions:   ['invoice', 'receipt', 'purchase_order', 'contract', 'leave_request', 'other', 'needs_human']
rewards:   {'correct': 1.0, 'abstain': 0.0, 'wrong': -1.0}

--- doc_01 ---
บริษัท สยามเปเปอร์ จำกัด
99/1 ถนนพระราม 4 กรุงเทพมหานคร 10110   เลขประจำตัวผู้เสียภาษี 0105551234567

ใบแจ้งหนี้
เลขที่ INV-2026-0042            วันที่ 1 ตุลาคม 2569
ลูกค้า: มหาวิทยาลัยศรีนครินทรวิโรฒ

รายการ                         จำนวน     ราคา/หน่วย      จำนวนเงิน
กระดาษ A4 80 แกรม              50 รีม      110.00       5,500.00
ภาษีมูลค่าเพิ่ม 7%                                         385.00
รวมทั้งสิ้น                                              5,885.00 บาท

กำหนดชำระ: ภายใน 30 วันนับจากวันที่ในเอกสาร



In [3]:
# The answer key. Agents never see this; only the environment uses it to score.
env.labels[["doc_type", "expected_action", "language", "tricky"]].head(8)

,doc_type,expected_action,language,tricky
doc_id,,,,
doc_01,invoice,invoice,th,0
doc_02,receipt,receipt,th,0
doc_03,purchase_order,purchase_order,en,0
doc_04,leave_request,leave_request,th,0
doc_05,contract,contract,th,0
doc_06,other,other,th,0
doc_07,invoice,invoice,en,0
doc_08,receipt,receipt,en,1


## 2. Simple reflex agent (10 นาที)

Agent แบบง่ายที่สุดใช้ **condition–action rule**: ถ้าเห็นคำนี้ ให้ทำสิ่งนี้ มันดูแค่ percept ปัจจุบัน ไม่จำอะไร และไม่คิดล่วงหน้า

Agent ด้านล่างมีกฎสำหรับ invoice เท่านั้น **งานของคุณ:** เพิ่มกฎสำหรับอีก 4 ประเภท (`receipt`, `purchase_order`, `contract`, `leave_request`) ทั้งคำไทยและคำอังกฤษ กฎถูกตรวจตามลำดับ กฎแรกที่ตรงชนะ

In [4]:
RULES = [
    # (keyword, action) - checked in order, first match wins
    ("ใบแจ้งหนี้", "invoice"),
    ("invoice", "invoice"),
    # TODO: add rules for receipt, purchase_order, contract and leave_request
]


class SimpleReflexAgent:
    """Condition-action rules on the current percept only."""

    def act(self, percept):
        text = percept.text.lower()
        for keyword, action in RULES:
            if keyword in text:
                return action
        return "other"


runs, agents = {}, {}
agents["simple reflex"] = SimpleReflexAgent()
runs["simple reflex"] = env.run(agents["simple reflex"])
summarize(runs, agents)

,reward,correct,abstain,wrong,seconds,llm_calls
agent,,,,,,
simple reflex,-12.0,6,0,18,0.0,0


ดูว่า agent พลาดฉบับไหน แล้วเปิดอ่านเอกสารนั้น 1–2 ฉบับ ถามตัวเองว่า: กฎข้อไหนทำให้พลาด และแก้ด้วยการเพิ่มกฎได้หรือไม่

In [5]:
def mistakes(name):
    df = runs[name]
    return df[df["outcome"] != "correct"][["expected", "action", "outcome"]]


def show(doc_id):
    print(next(p.text for p in env.percepts if p.doc_id == doc_id))


display(mistakes("simple reflex"))
show("doc_09")

,expected,action,outcome
doc_id,,,
doc_02,receipt,other,wrong
doc_03,purchase_order,invoice,wrong
doc_04,leave_request,other,wrong
doc_05,contract,other,wrong
doc_08,receipt,invoice,wrong
doc_09,other,invoice,wrong
doc_10,purchase_order,other,wrong
doc_11,contract,other,wrong
doc_12,leave_request,other,wrong


บันทึกข้อความ
ส่วนงาน ฝ่ายบัญชี
เรื่อง แนวปฏิบัติใหม่ในการส่งใบแจ้งหนี้และใบสั่งซื้อ

เรียน หัวหน้าฝ่ายทุกฝ่าย
ตั้งแต่วันที่ 1 พฤศจิกายน 2569 ขอให้ทุกฝ่ายแนบเลขที่ใบสั่งซื้อทุกครั้งที่ส่งใบแจ้งหนี้
ของผู้ขายมายังฝ่ายบัญชี เอกสารที่ไม่มีเลขที่ใบสั่งซื้อจะถูกส่งคืน

จึงเรียนมาเพื่อทราบและถือปฏิบัติ



## 3. Model-based reflex agent (10 นาที)

เอกสาร `doc_14` และ `doc_22` เป็น invoice ที่ **ถูกส่งซ้ำ** (เลขที่เดียวกับฉบับที่เข้ามาก่อนหน้า) ถ้าส่งต่อไปจ่ายเงินจะจ่ายซ้ำ action ที่ถูกจึงเป็น `needs_human`

Simple reflex agent ทำเรื่องนี้ไม่ได้ ไม่ว่าจะเพิ่มกฎกี่ข้อ เพราะการรู้ว่า "ซ้ำ" ต้องอาศัยสิ่งที่เคยเห็นมาก่อน **Model-based agent** เก็บ internal state ที่สรุปประวัติของ percept ไว้

**งานของคุณ:** เก็บเลขที่ invoice ที่เคยเห็นไว้ใน `self.seen` ถ้าเจอเลขที่ซ้ำให้คืน `needs_human`

In [6]:
import re

DOC_NO = re.compile(r"\b[A-Z]{2,3}-\d[\d-]*\b")  # matches INV-2026-0042, TI-55012, ...


class ModelBasedReflexAgent(SimpleReflexAgent):
    """Same rules, plus internal state: the invoice numbers seen so far."""

    def reset(self):
        self.seen = set()

    def act(self, percept):
        action = super().act(percept)
        if action == "invoice":
            match = DOC_NO.search(percept.text)
            # TODO: if this invoice number was seen before, return "needs_human";
            #       otherwise remember it in self.seen
        return action


agents["model-based reflex"] = ModelBasedReflexAgent()
runs["model-based reflex"] = env.run(agents["model-based reflex"])
summarize(runs, agents)

,reward,correct,abstain,wrong,seconds,llm_calls
agent,,,,,,
simple reflex,-12.0,6,0,18,0.0,0
model-based reflex,-12.0,6,0,18,0.0,0


## 4. LLM agent (15 นาที)

ตอนนี้ให้ LLM เป็นส่วนตัดสินใจ เราใช้ structured output บังคับให้คำตอบเป็น 1 ใน 6 ประเภท โปรแกรมจึงนำไปใช้ต่อได้ทันที

สังเกตว่า schema มี field `doc_no` ด้วย agent ตัวนี้ยังไม่ได้ใช้ แต่จะจำเป็นในส่วนถัดไป

**งานของคุณ:** `SYSTEM` prompt ด้านล่างบอกแค่ชื่อประเภท ให้เขียนคำนิยามสั้นๆ ของแต่ละประเภท โดยเฉพาะกรณีที่เอกสาร *พูดถึง* ประเภทอื่น (เช่น ใบเสร็จที่อ้างเลขที่ invoice หรือใบเสนอราคาที่หน้าตาเหมือน invoice)

<small>การรันเอกสาร 24 ฉบับใช้เวลาราว 30–60 วินาทีด้วย local model ระหว่างปรับ prompt ใช้ `InboxEnvironment(limit=10)` เพื่อให้รอบสั้นลงได้</small>

In [7]:
from typing import Literal

from pydantic import BaseModel, Field


class Routing(BaseModel):
    doc_type: Literal["invoice", "receipt", "purchase_order", "contract", "leave_request", "other"]
    doc_no: str | None = Field(description="The document's own number, exactly as printed; null if it has none")


# TODO: replace this with a prompt that defines each type in one line
SYSTEM = "Classify the document as one of: invoice, receipt, purchase_order, contract, leave_request, other."


class LLMAgent:
    """The LLM is the agent function: percept in, action out."""

    def __init__(self, llm):
        self.model = llm.with_structured_output(Routing)
        self.llm_calls = 0

    def reset(self):
        self.llm_calls = 0

    def decide(self, percept) -> Routing:
        self.llm_calls += 1
        return self.model.invoke([("system", SYSTEM), ("user", percept.text)])

    def act(self, percept):
        return self.decide(percept).doc_type


llm = get_llm()
agents["LLM"] = LLMAgent(llm)
runs["LLM"] = env.run(agents["LLM"])
display(summarize(runs, agents))
mistakes("LLM")

,reward,correct,abstain,wrong,seconds,llm_calls
agent,,,,,,
simple reflex,-12.0,6,0,18,0.0,0
model-based reflex,-12.0,6,0,18,0.0,0
LLM,18.0,21,0,3,383.7,24


,expected,action,outcome
doc_id,,,
doc_14,needs_human,invoice,wrong
doc_22,needs_human,invoice,wrong
doc_23,other,invoice,wrong


## 5. LLM + state (10 นาที)

LLM agent อ่านเอกสารเก่งกว่ากฎมาก แต่ดูรายการที่พลาด: มันยังพลาด invoice ซ้ำ เหตุผลเดียวกับ Module 0 คือ **LLM ไม่มี memory** แต่ละ request เริ่มจากศูนย์

ทางแก้คือสิ่งที่ทำในส่วนที่ 3: ให้ *โปรแกรมของเรา* เก็บ state แล้วใช้ LLM เป็นส่วนอ่านเอกสาร นี่คือโครงสร้างของ agent สมัยใหม่ทุกตัว: LLM ตัดสินใจ ส่วน code รอบๆ ดูแล state

**งานของคุณ:** ใช้ `doc_no` ที่ LLM ดึงออกมา ตรวจกับ `self.seen` เหมือนส่วนที่ 3

In [8]:
class StatefulLLMAgent(LLMAgent):
    """LLM for reading, program state for remembering."""

    def reset(self):
        super().reset()
        self.seen = set()

    def act(self, percept):
        decision = self.decide(percept)
        # TODO: if decision.doc_type is "invoice" and decision.doc_no was seen before,
        #       return "needs_human"; otherwise remember decision.doc_no
        return decision.doc_type


agents["LLM + state"] = StatefulLLMAgent(llm)
runs["LLM + state"] = env.run(agents["LLM + state"])
summarize(runs, agents)

,reward,correct,abstain,wrong,seconds,llm_calls
agent,,,,,,
simple reflex,-12.0,6,0,18,0.0,0
model-based reflex,-12.0,6,0,18,0.0,0
LLM,18.0,21,0,3,383.7,24
LLM + state,18.0,21,0,3,341.6,24


## 6. เปรียบเทียบ และดึงข้อมูลจากเอกสาร (15 นาที)

### 6.1 Agent ตัวไหน rational ที่สุด
คำตอบขึ้นกับ performance measure เราเพิ่ม agent อีกหนึ่งตัวก่อน: `CautiousReflexAgent` ใช้กฎชุดเดิม แต่ลงมือเฉพาะเมื่อมีประเภทเดียวที่ตรง นอกนั้นส่งให้คน

ตารางด้านล่างคิดคะแนนใหม่จากผลเดิม (ไม่ต้องรัน agent ซ้ำ) ภายใต้เกณฑ์สามแบบ ลองแก้ตัวเลขใน `measures` แล้วดูว่าอันดับเปลี่ยนหรือไม่ และอย่าลืมดูคอลัมน์ `seconds` กับ `llm_calls` ในตารางก่อนหน้า ซึ่งเป็นต้นทุนที่ performance measure ของเรายังไม่ได้นับ

In [9]:
def matching_types(text):
    text = text.lower()
    return {action for keyword, action in RULES if keyword in text}


class CautiousReflexAgent:
    """Acts only when exactly one type matches; otherwise hands the document to a person."""

    def act(self, percept):
        types = matching_types(percept.text)
        return next(iter(types)) if len(types) == 1 else "needs_human"


agents["cautious reflex"] = CautiousReflexAgent()
runs["cautious reflex"] = env.run(agents["cautious reflex"])

measures = {
    "default (+1, 0, -1)": {"correct": 1, "abstain": 0, "wrong": -1},
    "mistakes are costly (+1, 0, -10)": {"correct": 1, "abstain": 0, "wrong": -10},
    "people are costly (+1, -3, -1)": {"correct": 1, "abstain": -3, "wrong": -1},
}
pd.DataFrame({label: {name: df["outcome"].map(reward).sum() for name, df in runs.items()} for label, reward in measures.items()})

,"default (+1, 0, -1)","mistakes are costly (+1, 0, -10)","people are costly (+1, -3, -1)"
simple reflex,-12,-174,-12
model-based reflex,-12,-174,-12
LLM,18,-9,18
LLM + state,18,-9,18
cautious reflex,-3,-57,-48


### 6.2 ดึงข้อมูลจากเอกสารการเงิน
การรู้ประเภทเอกสารเป็นแค่ขั้นแรก งาน document intelligence ต้องดึงข้อมูลออกมาใช้ต่อ เราให้ LLM ดึง 4 field จากเอกสารการเงิน 11 ฉบับ แล้ววัดความถูกต้องทีละ field

สังเกตการแบ่งงาน: LLM **คัดลอกวันที่ตามที่พิมพ์ไว้** ส่วนการแปลงเป็นรูปแบบมาตรฐานเป็นหน้าที่ของ code เพราะ model ขนาดเล็กแปลงปี พ.ศ. เป็น ค.ศ. ผิดบ่อย

**งานของคุณ:** `to_iso_date` ด้านล่างรู้จักเฉพาะเดือนภาษาอังกฤษ เพิ่มเดือนภาษาไทย และแปลงปี พ.ศ. เป็น ค.ศ. (ลบ 543)

In [10]:
class FinancialDoc(BaseModel):
    doc_no: str = Field(description="The document's own number, exactly as printed")
    issue_date_text: str | None = Field(description="The date the document was issued, copied exactly as printed; null if it has none")
    total: float = Field(description="Final total amount including VAT")
    currency: Literal["THB", "USD", "EUR"]


MONTHS = {m: i for i, m in enumerate(
    ["january", "february", "march", "april", "may", "june", "july", "august", "september", "october", "november", "december"], 1)}
# TODO: add the Thai month names to MONTHS, for example MONTHS["ตุลาคม"] = 10


def to_iso_date(text):
    """'5 October 2026' -> '2026-10-05'. Returns '' when no date is found."""
    match = re.search(r"(\d{1,2})\s+(\S+)\s+(\d{4})", text or "")
    if not match or match.group(2).lower() not in MONTHS:
        return ""
    day, month, year = int(match.group(1)), MONTHS[match.group(2).lower()], int(match.group(3))
    # TODO: a year above 2400 is in the Buddhist era; subtract 543
    return f"{year:04d}-{month:02d}-{day:02d}"


financial = env.labels[env.labels["total"].notna() & (env.labels["expected_action"] != "needs_human")]
texts = {p.doc_id: p.text for p in env.percepts}
extractor = llm.with_structured_output(FinancialDoc)

rows = []
for doc_id in financial.index:
    doc = extractor.invoke([("system", "Extract the fields from this business document."), ("user", texts[doc_id])])
    rows.append({"doc_id": doc_id, "doc_no": doc.doc_no, "issue_date": to_iso_date(doc.issue_date_text),
                 "total": doc.total, "currency": doc.currency, "date_as_printed": doc.issue_date_text})
extracted = pd.DataFrame(rows).set_index("doc_id")
release_model()  # give the model's RAM back to the machine

truth = financial[["doc_no", "issue_date", "total", "currency"]].fillna({"issue_date": ""})
correct = pd.DataFrame({
    "doc_no": extracted["doc_no"].str.strip() == truth["doc_no"],
    "issue_date": extracted["issue_date"] == truth["issue_date"],
    "total": (extracted["total"] - truth["total"]).abs() < 0.005,
    "currency": extracted["currency"] == truth["currency"],
})
print("field accuracy:", correct.mean().round(2).to_dict())
extracted[~correct.all(axis=1)]

field accuracy: {'doc_no': 1.0, 'issue_date': 0.55, 'total': 1.0, 'currency': 1.0}


,doc_no,issue_date,total,currency,date_as_printed
doc_id,,,,,
doc_01,INV-2026-0042,,5885.0,THB,1 ตุลาคม 2569
doc_02,RC-0931,,225.0,THB,2 ตุลาคม 2569
doc_10,PO-2026-118,,18500.0,THB,7 ตุลาคม 2569
doc_15,RT-6690112,,1926.0,THB,10 ตุลาคม 2569
doc_18,PO-2026-121,2026-10-20,48000.0,THB,20 October 2026


## 7. Stretch: hybrid agent

LLM + state แม่นที่สุด แต่เรียก LLM ทุกฉบับ ทั้งที่เอกสารจำนวนมากง่ายพอที่กฎจะจัดการได้ **Hybrid agent** ใช้กฎก่อน แล้วเรียก LLM เฉพาะเมื่อกฎไม่แน่ใจ นี่คือแนวคิด System 1 (เร็ว ถูก) กับ System 2 (ช้า รอบคอบ) ที่เราจะกลับมาใน Module 3

`matching_types` จากส่วนที่ 6 คืนประเภททั้งหมดที่มีกฎตรง **งานของคุณ:** ถ้าตรงประเภทเดียวพอดี ให้ใช้ประเภทนั้นโดยไม่เรียก LLM นอกนั้นให้ถาม LLM แล้วเทียบคะแนนกับจำนวน `llm_calls`

ถ้าเหลือเวลา ลองเพิ่มเติม:
- เปิด thinking ด้วย `get_llm(reasoning=True)` แล้วเทียบคะแนนกับเวลา
- ถ้ามี API key ลอง `get_llm("openrouter")` กับ agent ตัวเดิม

In [11]:
class HybridAgent(StatefulLLMAgent):
    """Rules when they are sure, the LLM when they are not."""

    def act(self, percept):
        types = matching_types(percept.text)
        # TODO: if exactly one type matches and it is not "invoice", return it without calling the LLM
        return super().act(percept)


agents["hybrid"] = HybridAgent(llm)
runs["hybrid"] = env.run(agents["hybrid"])
summarize(runs, agents)

,reward,correct,abstain,wrong,seconds,llm_calls
agent,,,,,,
simple reflex,-12.0,6,0,18,0.0,0
model-based reflex,-12.0,6,0,18,0.0,0
LLM,18.0,21,0,3,383.7,24
LLM + state,18.0,21,0,3,341.6,24
cautious reflex,-3.0,3,15,6,0.0,0
hybrid,18.0,21,0,3,254.8,24


## สรุปและสิ่งที่ต้องส่ง

ตอบคำถาม 4 ข้อนี้ ข้อละ 2–3 ประโยค แล้วส่งพร้อมภาพตารางสรุปจากส่วนที่ 5 ตามที่ผู้สอนกำหนด

1. Simple reflex agent พลาดเอกสารแบบไหน และทำไมการเพิ่มกฎจึงแก้ได้ไม่หมด
2. ทำไม LLM agent ที่อ่านเอกสารได้ดีจึงยังพลาด invoice ซ้ำ และอะไรแก้ปัญหานี้ได้
3. เทียบ simple reflex กับ cautious reflex ภายใต้เกณฑ์ทั้งสามแบบ ตัวไหนดีกว่าในแต่ละเกณฑ์ ข้อนี้บอกอะไรเกี่ยวกับการออกแบบ performance measure
4. ในส่วน 6.2 field ไหนที่ LLM ดึงได้แม่น และ field ไหนที่ต้องให้ code ช่วย

**จาก notebook สู่โปรแกรม:** เมื่อทำ notebook จบแล้ว เปิด `inbox_app.py` ในโฟลเดอร์เดียวกัน นั่นคือ agent ตัวนี้ที่เขียนแบบนำไปรันจริง มี state ที่บันทึกลงไฟล์ ผลลัพธ์เป็น CSV และการรับมือเมื่อ LLM ล้มเหลว ดูวิธีรันในคู่มือ Lab 1

**Project milestone 1:** กลุ่มของคุณจะใช้คำตอบเหล่านี้ตอนเขียน PEAS และเลือกโครงสร้าง agent สำหรับปัญหาเอกสารของกลุ่ม